# WTI-Brent Crude Oil Price Spread Analysis

## 1. Project Overview

This project analyzes the historical price relationship between West Texas Intermediate (WTI) and Brent crude oil benchmarks.

The primary objective is to investigate long-term spread trends, identify unusual spread movements, examine periods of persistent deviations, and understand how changes in the two benchmark prices contribute to spread fluctuations.

## 2. Data Preparation

Daily WTI and Brent price datasets were loaded and merged using an inner join on the Date column.

The WTI-Brent spread was calculated as:

**Spread = WTI Price - Brent Price**

The resulting dataset contains approximately 9,790 observations.

### Data Source

This project uses daily crude oil spot price data for West Texas Intermediate (WTI) and Brent crude oil.

The data were obtained from the DataHub dataset *Crude Oil Spot Prices — Brent and WTI*, compiled by DataPressr from the U.S. Energy Information Administration (EIA).

- **WTI:** Cushing, Oklahoma WTI Spot Price FOB
- **Brent:** Europe Brent Spot Price FOB
- **Unit:** U.S. dollars per barrel (USD/bbl)
- **Frequency:** Daily
- **Variables:** Date and Price
- **Source:** U.S. Energy Information Administration (EIA)
- **Data access:** https://datahub.io/datapressr/oil-prices
- **License:** Open Data Commons Public Domain Dedication and License (PDDL-1.0)

The two datasets were merged using an inner join on the Date column. The analysis therefore includes dates for which both WTI and Brent observations are available.

The WTI-Brent spread was calculated as:

\[
Spread = WTI - Brent
\]

A negative spread indicates that WTI traded below Brent, while a positive spread indicates that WTI traded above Brent.

**Source acknowledgement:** U.S. Energy Information Administration (EIA), crude oil spot price data, accessed through DataHub/DataPressr. Data snapshot retrieved September 5, 2026.

### Reproducibility

Place `wti-daily.csv` and `brent-daily.csv` in the same folder as this notebook before running it from top to bottom. The WTI source file was not included with the materials available for this handoff, so a fresh end-to-end execution has not been performed. Existing saved outputs are retained except for the corrected outlier-run cells, whose outputs were cleared so they can be regenerated.


In [ ]:
import pandas as pd 
from IPython.display import display
from plotly.subplots import make_subplots
import plotly.graph_objects as go

df_wti = pd.read_csv("data/wti-daily.csv")
df_brent = pd.read_csv("data/brent-daily.csv")

print("WTI and Brent DataFrame head:")
display(df_wti.head())
display(df_brent.head())

print("WTI and Brent DataFrame shape:")
display(df_wti.shape)
display(df_brent.shape)

print("WTI and Brent DataFrame info:")
df_wti.info()
df_brent.info()

print("WTI and Brent DataFrame describe:")
display(df_wti.describe())
display(df_brent.describe())

WTI and Brent DataFrame head:


,Date,Price
0,1986-01-02,25.56
1,1986-01-03,26.00
2,1986-01-06,26.53
3,1986-01-07,25.85
4,1986-01-08,25.87


,Date,Price
0,1987-05-20,18.63
1,1987-05-21,18.45
2,1987-05-22,18.55
3,1987-05-25,18.60
4,1987-05-26,18.63


WTI and Brent DataFrame shape:


(10236, 2)

(9967, 2)

WTI and Brent DataFrame info:
<class 'pandas.DataFrame'>
RangeIndex: 10236 entries, 0 to 10235
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    10236 non-null  str    
 1   Price   10236 non-null  float64
dtypes: float64(1), str(1)
memory usage: 160.1 KB
<class 'pandas.DataFrame'>
RangeIndex: 9967 entries, 0 to 9966
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    9967 non-null   str    
 1   Price   9967 non-null   float64
dtypes: float64(1), str(1)
memory usage: 155.9 KB
WTI and Brent DataFrame describe:


,Price
count,10236.000000
mean,48.631400
std,29.605325
min,-36.980000
25%,20.380000
50%,43.865000
75%,71.765000
max,145.310000


,Price
count,9967.000000
mean,51.437985
std,32.791572
min,9.100000
25%,19.600000
50%,46.610000
75%,75.095000
max,143.950000


## 3. Methodology

The analysis includes:

- Data quality checks, including missing values and duplicate records.
- Descriptive statistics of the WTI-Brent spread.
- Interquartile Range (IQR) method for outlier detection.
- Time-series analysis using monthly and annual averages.
- Rolling average analysis to identify longer-term trends.
- Outlier period identification and duration analysis.
- Interactive dashboard development using Python and Plotly.

### Data quality checks, including missing values and duplicate records.

In [65]:
print("converting 'Date' column to datetime format:")
df_wti['Date'] = pd.to_datetime(df_wti['Date'], format='%Y-%m-%d', errors='coerce')
df_brent['Date'] = pd.to_datetime(df_brent['Date'], format='%Y-%m-%d', errors='coerce')

print("WTI and Brent DataFrame duplicated rows count:")
display(df_wti.duplicated().sum())
display(df_brent.duplicated().sum())

print("check if date is ordered ascending:")
display(df_wti['Date'].is_monotonic_increasing)
display(df_brent['Date'].is_monotonic_increasing)

print("check if price has missing values:")
display(df_wti['Price'].isna().sum())
display(df_brent['Price'].isna().sum())

print("which date that wti and brent price is the lowest:")
display(df_wti.loc[df_wti['Price'].idxmin()])
display(df_brent.loc[df_brent['Price'].idxmin()])

converting 'Date' column to datetime format:
WTI and Brent DataFrame duplicated rows count:


np.int64(0)

np.int64(0)

check if date is ordered ascending:


True

True

... (2 outputs left out)

np.int64(0)

which date that wti and brent price is the lowest:


Date     2020-04-20 00:00:00
Price                 -36.98
Name: 8643, dtype: object

Date     1998-12-10 00:00:00
Price                    9.1
Name: 2937, dtype: object

In [66]:
print("check if Date column is converted to datetime format:")
display(df_wti['Date'].dtype)
display(df_brent['Date'].dtype)

check if Date column is converted to datetime format:


dtype('<M8[us]')

dtype('<M8[us]')

In [67]:
print("rename column Price in both dataframes to prepare for merge:")
df_wti.rename(columns={'Price':'WTI'}, inplace=True)
df_brent.rename(columns={'Price':'Brent'}, inplace=True)
display(df_wti.head())
display(df_brent.head())

rename column Price in both dataframes to prepare for merge:


,Date,WTI
0,1986-01-02,25.56
1,1986-01-03,26.00
2,1986-01-06,26.53
3,1986-01-07,25.85
4,1986-01-08,25.87


,Date,Brent
0,1987-05-20,18.63
1,1987-05-21,18.45
2,1987-05-22,18.55
3,1987-05-25,18.60
4,1987-05-26,18.63


In [68]:
# merge the two dataframes on the 'Date' column
df_oil = pd.merge(df_wti, df_brent, on='Date', how='inner')

display(df_oil.shape)
display(df_oil.head())
display(df_oil.dtypes)
display(df_oil.isna().sum())
display(df_oil.duplicated().sum())
display(df_oil['Date'].duplicated().sum())
display(df_oil['Date'].is_monotonic_increasing)


(9790, 3)

,Date,WTI,Brent
0,1987-05-20,19.75,18.63
1,1987-05-21,19.95,18.45
2,1987-05-22,19.68,18.55
3,1987-05-26,19.35,18.63
4,1987-05-27,19.38,18.60


Date     datetime64[us]
WTI             float64
Brent           float64
dtype: object

Date     0
WTI      0
Brent    0
dtype: int64

np.int64(0)

np.int64(0)

True

In [69]:
# check first and last date in the merged dataframe
display(df_oil.head(1)['Date'])
display(df_oil.tail(1)['Date'])

0   1987-05-20
Name: Date, dtype: datetime64[us]

9789   2026-09-01
Name: Date, dtype: datetime64[us]

### Descriptive statistics of the WTI-Brent spread.


In [70]:
# add spread = wti - brent
df_oil['Spread'] = df_oil['WTI'] - df_oil['Brent']
display(df_oil.head())

,Date,WTI,Brent,Spread
0,1987-05-20,19.75,18.63,1.12
1,1987-05-21,19.95,18.45,1.50
2,1987-05-22,19.68,18.55,1.13
3,1987-05-26,19.35,18.63,0.72
4,1987-05-27,19.38,18.60,0.78


In [71]:
# get statistics of the spread
display(df_oil['Spread'].mean())
display(df_oil['Spread'].median())
display(df_oil['Spread'].min())
display(df_oil['Spread'].max())

np.float64(-1.7080704800817161)

np.float64(0.629999999999999)

np.float64(-54.339999999999996)

np.float64(22.179999999999993)

In [72]:
# find out the line when the spread is the smallest
min_spread = df_oil.loc[df_oil['Spread'].idxmin()]
print(min_spread)

Date      2020-04-20 00:00:00
WTI                    -36.98
Brent                   17.36
Spread                 -54.34
Name: 8226, dtype: object


In [73]:
# find out the line when the spread is the largest
max_spread = df_oil.loc[df_oil['Spread'].idxmax()]
print(max_spread)

Date      2008-09-22 00:00:00
WTI                    122.61
Brent                  100.43
Spread                  22.18
Name: 5330, dtype: object


In [74]:
display((df_oil['Spread'] < 0).sum())
display((df_oil['Spread'] > 0).sum())
display((df_oil['Spread'] == 0).sum())

np.int64(4379)

np.int64(5408)

np.int64(3)

### Interquartile Range (IQR) method for outlier detection.


In [75]:
display(df_oil['Spread'].std())
q1 = df_oil['Spread'].quantile(0.25)
q3 = df_oil['Spread'].quantile(0.75)
iqr = q3 - q1
print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)


np.float64(5.391948167546939)

Q1: -3.757499999999995
Q3: 1.6099999999999994
IQR: 5.367499999999994


In [76]:
display(df_oil['Spread'].describe())

count    9790.000000
mean       -1.708070
std         5.391948
min       -54.340000
25%        -3.757500
50%         0.630000
75%         1.610000
max        22.180000
Name: Spread, dtype: float64

In [77]:
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

display((df_oil['Spread'] < lower_bound).sum())
display((df_oil['Spread'] > upper_bound).sum())

Lower Bound: -11.808749999999987
Upper Bound: 9.661249999999992


np.int64(559)

np.int64(4)

In [78]:
df_spread_lower_outliers = df_oil[df_oil['Spread'] < lower_bound]
display(df_spread_lower_outliers)
display(df_spread_lower_outliers.head())
display(df_spread_lower_outliers.shape)
display(df_spread_lower_outliers.sort_values('Date'))

,Date,WTI,Brent,Spread
5429,2009-02-12,34.03,47.23,-13.20
5922,2011-01-27,84.45,96.48,-12.03
5927,2011-02-03,89.42,101.69,-12.27
5929,2011-02-07,86.30,99.44,-13.14
5930,2011-02-08,85.85,99.25,-13.40
...,...,...,...,...
9704,2026-04-28,103.45,117.62,-14.17
9705,2026-04-29,110.47,124.16,-13.69
9706,2026-04-30,108.64,124.24,-15.60
9707,2026-05-01,105.38,118.26,-12.88


,Date,WTI,Brent,Spread
5429,2009-02-12,34.03,47.23,-13.20
5922,2011-01-27,84.45,96.48,-12.03
5927,2011-02-03,89.42,101.69,-12.27
5929,2011-02-07,86.30,99.44,-13.14
5930,2011-02-08,85.85,99.25,-13.40


(559, 4)

,Date,WTI,Brent,Spread
5429,2009-02-12,34.03,47.23,-13.20
5922,2011-01-27,84.45,96.48,-12.03
5927,2011-02-03,89.42,101.69,-12.27
5929,2011-02-07,86.30,99.44,-13.14
5930,2011-02-08,85.85,99.25,-13.40
...,...,...,...,...
9704,2026-04-28,103.45,117.62,-14.17
9705,2026-04-29,110.47,124.16,-13.69
9706,2026-04-30,108.64,124.24,-15.60
9707,2026-05-01,105.38,118.26,-12.88


In [79]:
df_spread_lower_outliers['Date'].dt.year.value_counts().sort_index()


Date
2009      1
2011    182
2012    222
2013     92
2014     17
2015      3
2018      1
2019      2
2020      1
2022      5
2026     33
Name: count, dtype: int64

In [80]:
df_outliers_2011_2013 = df_spread_lower_outliers[df_spread_lower_outliers['Date'].dt.year.isin(range(2011,2014))]
df_outliers_2011_2013[['Date','Spread']]

,Date,Spread
5922,2011-01-27,-12.03
5927,2011-02-03,-12.27
5929,2011-02-07,-13.14
5930,2011-02-08,-13.40
5931,2011-02-09,-14.57
...,...,...
6647,2013-12-20,-13.04
6648,2013-12-23,-12.96
6649,2013-12-24,-12.70
6650,2013-12-26,-12.47


In [81]:
df_yearly_counts = df_oil['Date'].dt.year.value_counts().sort_index()
df_yearly_outliers_counts = df_spread_lower_outliers['Date'].dt.year.value_counts().sort_index()
df_yearly_counts_merged = pd.merge(
    df_yearly_counts, 
    df_yearly_outliers_counts, 
    left_index=True, 
    right_index=True, 
    how='outer', 
    suffixes=('_total', '_outliers'))

# fillna(0) 将 NaN 替换为 0，astype(int) 将数据类型转换为整数类型。
df_yearly_counts_merged['count_outliers'] = (
    df_yearly_counts_merged['count_outliers']
    .fillna(0)
    .astype(int))

# value_counts() 返回的是一个 Series，它的索引名称仍然是 Date，
# 而 pd.merge() 合并后，Date 作为索引显示在单独的表头行里。
# 最简单的改法：在合并后使用 reset_index()，把年份索引转换成普通列。
df_yearly_counts_merged = df_yearly_counts_merged.reset_index()

df_yearly_counts_merged = df_yearly_counts_merged.rename(
    columns={'Date': 'Year'}
)
display(df_yearly_counts)
display(df_yearly_counts_merged)

Date
1987    158
1988    254
1989    254
1990    256
1991    256
1992    256
1993    245
1994    247
1995    248
1996    250
1997    244
1998    247
1999    246
2000    247
2001    249
2002    246
2003    250
2004    249
2005    249
2006    247
2007    250
2008    253
2009    252
2010    252
2011    248
2012    249
2013    252
2014    252
2015    251
2016    251
2017    249
2018    246
2019    250
2020    249
2021    246
2022    245
2023    243
2024    246
2025    244
2026    164
Name: count, dtype: int64

,Year,count_total,count_outliers
0,1987,158,0
1,1988,254,0
2,1989,254,0
3,1990,256,0
4,1991,256,0
5,1992,256,0
6,1993,245,0
7,1994,247,0
8,1995,248,0
9,1996,250,0


In [82]:
# 更推荐的写法是在统计时就明确给 Series 命名，然后使用 to_frame() 转成 DataFrame，这样列名和索引更清楚
df_yearly_counts = (
    df_oil['Date'].dt.year
    .value_counts()
    .sort_index()
    .rename('count_total')
    .to_frame()
)
display(df_yearly_counts)

df_yearly_outliers_counts = (
    df_spread_lower_outliers['Date'].dt.year
    .value_counts()
    .sort_index()
    .rename('count_outliers')
    .to_frame()
)

display(df_yearly_outliers_counts)

df_yearly_counts_merged = df_yearly_counts.join(
    df_yearly_outliers_counts,
    how='left'
).fillna({'count_outliers': 0})

df_yearly_counts_merged['count_outliers'] = (
    df_yearly_counts_merged['count_outliers'].astype(int)
)

df_yearly_counts_merged.index.name = 'Year'
df_yearly_counts_merged = df_yearly_counts_merged.reset_index()

display(df_yearly_counts_merged)

,count_total
Date,
1987,158
1988,254
1989,254
1990,256
1991,256
1992,256
1993,245
1994,247
1995,248


,count_outliers
Date,
2009,1
2011,182
2012,222
2013,92
2014,17
2015,3
2018,1
2019,2
2020,1


,Year,count_total,count_outliers
0,1987,158,0
1,1988,254,0
2,1989,254,0
3,1990,256,0
4,1991,256,0
5,1992,256,0
6,1993,245,0
7,1994,247,0
8,1995,248,0
9,1996,250,0


In [83]:
df_yearly_counts_merged['outlier_pct'] = (
    df_yearly_counts_merged['count_outliers']
    / df_yearly_counts_merged['count_total']
    * 100
)

display(df_yearly_counts_merged.style.format({
    'outlier_pct': '{:.2f}'
}))

,Year,count_total,count_outliers,outlier_pct
0,1987,158,0,0.00
1,1988,254,0,0.00
2,1989,254,0,0.00
3,1990,256,0,0.00
4,1991,256,0,0.00
5,1992,256,0,0.00
6,1993,245,0,0.00
7,1994,247,0,0.00
8,1995,248,0,0.00
9,1996,250,0,0.00


In [84]:
# 使用 Pandas 自带的显示选项
with pd.option_context('display.float_format', '{:.2f}'.format):
    display(df_yearly_counts_merged)

,Year,count_total,count_outliers,outlier_pct
0,1987,158,0,0.00
1,1988,254,0,0.00
2,1989,254,0,0.00
3,1990,256,0,0.00
4,1991,256,0,0.00
5,1992,256,0,0.00
6,1993,245,0,0.00
7,1994,247,0,0.00
8,1995,248,0,0.00
9,1996,250,0,0.00


In [85]:
df_outliers_2011_2013 = df_spread_lower_outliers[df_spread_lower_outliers['Date'].dt.year.isin(range(2011,2014))]
df_outliers_2011_2013[['Date','Spread']]

,Date,Spread
5922,2011-01-27,-12.03
5927,2011-02-03,-12.27
5929,2011-02-07,-13.14
5930,2011-02-08,-13.40
5931,2011-02-09,-14.57
...,...,...
6647,2013-12-20,-13.04
6648,2013-12-23,-12.96
6649,2013-12-24,-12.70
6650,2013-12-26,-12.47


In [86]:
df_yearly_counts_merged.sort_values(
    'outlier_pct', 
    ascending=False
).head(10)


,Year,count_total,count_outliers,outlier_pct
25,2012,249,222,89.156627
24,2011,248,182,73.387097
26,2013,252,92,36.507937
39,2026,164,33,20.121951
27,2014,252,17,6.746032
35,2022,245,5,2.040816
28,2015,251,3,1.195219
32,2019,250,2,0.800000
31,2018,246,1,0.406504
33,2020,249,1,0.401606


In [87]:
print(df_yearly_counts_merged.columns)

Index(['Year', 'count_total', 'count_outliers', 'outlier_pct'], dtype='str')


In [88]:
import plotly.express as px

fig = px.line(
    df_oil,
    x="Date",
    y="Spread",
    title="WTI-Brent Spread Over Time"
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Spread (USD/barrel)"
)

# 在 px.line() 中，x="Date" 和 y="Spread" 是列名字符串；
# 但在 fig.add_scatter() 中，x 和 y 需要传入实际的数据，而不是列名字符串。
fig.add_scatter(
    x=df_spread_lower_outliers["Date"],
    y=df_spread_lower_outliers["Spread"],
    mode="markers",
    name="Lower Outliers"
)

# IQR 下界参考线
fig.add_hline(
    y=-11.80875,
    line_dash="dash",
    line_color="red",
    annotation_text="Lower IQR Bound",
    annotation_position="bottom right"
)

# 参数	作用
# x0, x1	阴影的起始和结束日期
# fillcolor	阴影颜色
# opacity	透明度，0 表示完全透明，1 表示不透明
# layer="below"	将阴影放在数据曲线下方
# line_width=0	不显示阴影区域的边框
# annotation_text	在阴影区域添加文字
fig.add_vrect(
    x0="2011-01-01",
    x1="2013-12-31",
    fillcolor="orange",
    opacity=0.15,
    layer="below",
    line_width=0,
    annotation_text="2011–2013",
    annotation_position="top left"
)

fig.show()

In [89]:
df_oil_2011_2013 = df_oil[
    (df_oil["Date"] >= "2011-01-01") 
    & (df_oil["Date"] <= "2013-12-31")
]

print(df_oil_2011_2013.shape)
display(df_oil_2011_2013.head())

(749, 4)


,Date,WTI,Brent,Spread
5905,2011-01-03,91.59,95.82,-4.23
5906,2011-01-04,89.39,93.52,-4.13
5907,2011-01-05,90.30,95.07,-4.77
5908,2011-01-06,88.37,94.95,-6.58
5909,2011-01-07,88.07,94.25,-6.18


In [90]:
fig_zoom = px.line(
    df_oil_2011_2013,
    x="Date",
    y="Spread",
    title="WTI-Brent Spread (2011-2013)"
)

fig_zoom.add_hline(
    y=-11.80875,
    line_dash="dash",
    line_color="red",
    annotation_text="Lower IQR Bound"
)

fig_zoom.show()

In [91]:
df_oil_2011_2013['is_lower_outlier'] = (
    df_oil_2011_2013['Spread'] < -11.80875
)
display(df_oil_2011_2013.head())
print(df_oil_2011_2013['is_lower_outlier'].value_counts())

,Date,WTI,Brent,Spread,is_lower_outlier
5905,2011-01-03,91.59,95.82,-4.23,False
5906,2011-01-04,89.39,93.52,-4.13,False
5907,2011-01-05,90.30,95.07,-4.77,False
5908,2011-01-06,88.37,94.95,-6.58,False
5909,2011-01-07,88.07,94.25,-6.18,False


is_lower_outlier
True     496
False    253
Name: count, dtype: int64


In [ ]:
df_oil_2011_2013['previous_outlier'] = (
    df_oil_2011_2013['is_lower_outlier']
    .shift(1)
    .fillna(False)
    .astype(bool)
)

display(
    df_oil_2011_2013[
        ['Date', 'Spread', 'is_lower_outlier', 'previous_outlier']
    ].head(10)
)

display(df_oil_2011_2013)

In [ ]:
df_oil_2011_2013['outlier_start'] = (
    df_oil_2011_2013['is_lower_outlier']
    & ~df_oil_2011_2013['previous_outlier']
)

display(df_oil_2011_2013)

display(
    df_oil_2011_2013.loc[
        df_oil_2011_2013['outlier_start'],
        ['Date', 'Spread', 'is_lower_outlier', 'previous_outlier', 'outlier_start']
    ]
)

In [94]:
df_oil_2011_2013['outlier_group'] = (
    df_oil_2011_2013['outlier_start'].cumsum()
)

df_oil_2011_2013

,Date,WTI,Brent,Spread,is_lower_outlier,previous_outlier,outlier_start,outlier_group
5905,2011-01-03,91.59,95.82,-4.23,False,True,False,0
5906,2011-01-04,89.39,93.52,-4.13,False,False,False,0
5907,2011-01-05,90.30,95.07,-4.77,False,False,False,0
5908,2011-01-06,88.37,94.95,-6.58,False,False,False,0
5909,2011-01-07,88.07,94.25,-6.18,False,False,False,0
...,...,...,...,...,...,...,...,...
6649,2013-12-24,98.87,111.57,-12.70,True,True,False,18
6650,2013-12-26,99.18,111.65,-12.47,True,True,False,18
6651,2013-12-27,99.94,112.06,-12.12,True,True,False,18
6652,2013-12-30,98.90,110.47,-11.57,False,True,False,18


In [95]:


df_outlier_periods = (
    df_oil_2011_2013.loc[df_oil_2011_2013['is_lower_outlier']]
    .groupby('outlier_group')
    .agg(
        start_date=('Date','min'),
        end_date=('Date','max'),
        duration=('Date','count'),
        min_spread=('Spread','min'),
        mean_spread=('Spread','mean')
    ).reset_index()
)

#display(df_outlier_periods)

df_outlier_periods['severity'] = (
    lower_bound - df_outlier_periods['mean_spread']
)

display(
    df_outlier_periods.sort_values(
        'duration', ascending=False
    ).head(10)
)
display(
    df_outlier_periods.sort_values(
        'severity', ascending=False
    ).head(10)
)

,outlier_group,start_date,end_date,duration,min_spread,mean_spread,severity
11,12,2012-07-03,2013-03-25,183,-24.87,-19.442732,7.633982
6,7,2011-04-04,2011-11-15,155,-29.59,-19.928774,8.120024
8,9,2012-02-01,2012-06-18,93,-21.51,-16.764839,4.956089
15,16,2013-11-12,2013-12-09,19,-19.27,-15.730526,3.921776
2,3,2011-02-07,2011-03-03,18,-19.46,-15.680000,3.871250
17,18,2013-12-13,2013-12-27,10,-13.12,-12.449000,0.640250
4,5,2011-03-15,2011-03-21,5,-13.88,-13.038000,1.229250
13,14,2013-04-02,2013-04-04,3,-12.80,-12.353333,0.544583
1,2,2011-02-03,2011-02-03,1,-12.27,-12.270000,0.461250
0,1,2011-01-27,2011-01-27,1,-12.03,-12.030000,0.221250


,outlier_group,start_date,end_date,duration,min_spread,mean_spread,severity
6,7,2011-04-04,2011-11-15,155,-29.59,-19.928774,8.120024
11,12,2012-07-03,2013-03-25,183,-24.87,-19.442732,7.633982
8,9,2012-02-01,2012-06-18,93,-21.51,-16.764839,4.956089
15,16,2013-11-12,2013-12-09,19,-19.27,-15.730526,3.921776
2,3,2011-02-07,2011-03-03,18,-19.46,-15.680000,3.871250
7,8,2012-01-12,2012-01-12,1,-13.94,-13.940000,2.131250
10,11,2012-06-28,2012-06-28,1,-13.30,-13.300000,1.491250
4,5,2011-03-15,2011-03-21,5,-13.88,-13.038000,1.229250
3,4,2011-03-11,2011-03-11,1,-12.93,-12.930000,1.121250
17,18,2013-12-13,2013-12-27,10,-13.12,-12.449000,0.640250


In [96]:
print(df_outlier_periods['duration'].sum())

496


In [97]:
df_major_periods = df_outlier_periods[
    df_outlier_periods['duration'] >= 90
]

display(df_major_periods)

,outlier_group,start_date,end_date,duration,min_spread,mean_spread,severity
6,7,2011-04-04,2011-11-15,155,-29.59,-19.928774,8.120024
8,9,2012-02-01,2012-06-18,93,-21.51,-16.764839,4.956089
11,12,2012-07-03,2013-03-25,183,-24.87,-19.442732,7.633982


In [98]:

for _, row in df_major_periods.iterrows():
    fig.add_vrect(
        x0=row['start_date'],
        x1=row['end_date'],
        fillcolor='orange',
        opacity=0.15,
        layer='below',
        line_width=0,
        annotation_text=f"Group {row['outlier_group']}"
    )


fig.show()

In [99]:
display(
    df_major_periods[
        [
            'outlier_group',
            'start_date',
            'end_date',
            'duration',
            'min_spread',
            'mean_spread',
            'severity'
        ]
    ].sort_values('duration', ascending=False)
)

,outlier_group,start_date,end_date,duration,min_spread,mean_spread,severity
11,12,2012-07-03,2013-03-25,183,-24.87,-19.442732,7.633982
6,7,2011-04-04,2011-11-15,155,-29.59,-19.928774,8.120024
8,9,2012-02-01,2012-06-18,93,-21.51,-16.764839,4.956089


### Time-series analysis using monthly and annual averages.


In [100]:
df_oil_monthly=(
    df_oil
    .set_index('Date')
    .resample('MS')
    .mean()
)

display(df_oil_monthly.head())

,WTI,Brent,Spread
Date,,,
1987-05-01,19.535714,18.577143,0.958571
1987-06-01,20.073333,18.860476,1.212857
1987-07-01,21.342174,19.856522,1.485652
1987-08-01,20.310952,18.979524,1.331429
1987-09-01,19.530000,18.313182,1.216818


### - Rolling average analysis to identify longer-term trends.


In [101]:
df_oil_monthly['Spread_12M_MA']=(
    df_oil_monthly['Spread']
    .rolling(window=12)
    .mean()
)

display(df_oil_monthly.head(15))

,WTI,Brent,Spread,Spread_12M_MA
Date,,,,
1987-05-01,19.535714,18.577143,0.958571,NaN
1987-06-01,20.073333,18.860476,1.212857,NaN
1987-07-01,21.342174,19.856522,1.485652,NaN
1987-08-01,20.310952,18.979524,1.331429,NaN
1987-09-01,19.530000,18.313182,1.216818,NaN
1987-10-01,19.859091,18.757727,1.101364,NaN
1987-11-01,18.854000,17.783500,1.070500,NaN
1987-12-01,17.274545,17.054091,0.220455,NaN
1988-01-01,17.129444,16.749444,0.380000,NaN


In [102]:
import plotly.express as px

fig = px.line(
    df_oil_monthly,
    x=df_oil_monthly.index,
    y='Spread',
    title='Monthly Average WTI-Brent Spread'
)

fig.add_scatter(
    x=df_oil_monthly.index,
    y=df_oil_monthly['Spread_12M_MA'],
    mode='lines',
    name='12-Month Rolling Average'
)

fig.update_layout(
    xaxis_title='Date',
    yaxis_title='Spread (USD/barrel)'
)

fig.show()

### 计算每年的平均价差

In [103]:
df_oil_yearly=(
    df_oil
    .set_index('Date')
    .resample('YS')
    .mean()
)

display(df_oil_yearly[['Spread']].head(10))

,Spread
Date,
1987-01-01,1.085886
1988-01-01,1.043425
1989-01-01,1.390669
1990-01-01,0.791133
1991-01-01,1.502617
1992-01-01,1.257461
1993-01-01,1.401265
1994-01-01,1.356964
1995-01-01,1.401290


In [104]:
fig = px.line(
    df_oil_yearly,
    x=df_oil_yearly.index,
    y='Spread',
    title='Annual Average WTI-Brent Spread'
)

fig.add_hline(
    y=0,
    line_dash='dash'
)

fig.show()

In [105]:
spread_2010 = df_oil_yearly.loc['2010-01-01','Spread']
spread_2011 = df_oil_yearly.loc['2011-01-01','Spread']
change = spread_2011 - spread_2010

print(spread_2010, spread_2011, change)

-0.13373015873015898 -16.520927419354837 -16.387197260624678


In [106]:
display(
    df_oil_yearly[['Spread']]
    .loc['2007':'2015']
)

,Spread
Date,
2007-01-01,-0.151440
2008-01-01,2.727154
2009-01-01,0.211667
2010-01-01,-0.133730
2011-01-01,-16.520927
2012-01-01,-17.606386
2013-01-01,-10.572460
2014-01-01,-5.717698
2015-01-01,-3.656693


In [107]:
## diff()。它会计算当前行与上一行的差值，不需要自己写循环。
## previous line 之前用的 shift(1)

df_oil_yearly['Spread_Change']=(
    df_oil_yearly['Spread'].diff()
)

df_oil_yearly.head()

,WTI,Brent,Spread,Spread_Change
Date,,,,
1987-01-01,19.616266,18.530380,1.085886,NaN
1988-01-01,15.952205,14.908780,1.043425,-0.042461
1989-01-01,19.618898,18.228228,1.390669,0.347244
1990-01-01,24.552578,23.761445,0.791133,-0.599536
1991-01-01,21.541367,20.038750,1.502617,0.711484


In [108]:
# 观察 2007–2015 年的年度价差及其变化
display(
    df_oil_yearly[['Spread', 'Spread_Change']]
    .loc['2007':'2015']
)

,Spread,Spread_Change
Date,,
2007-01-01,-0.151440,-1.073262
2008-01-01,2.727154,2.878594
2009-01-01,0.211667,-2.515487
2010-01-01,-0.133730,-0.345397
2011-01-01,-16.520927,-16.387197
2012-01-01,-17.606386,-1.085458
2013-01-01,-10.572460,7.033925
2014-01-01,-5.717698,4.854762
2015-01-01,-3.656693,2.061005


In [109]:
spread_change_min_year=df_oil_yearly['Spread_Change'].idxmin()
spread_change_max_year=df_oil_yearly['Spread_Change'].idxmax()

print(spread_change_min_year, spread_change_max_year)

2011-01-01 00:00:00 2013-01-01 00:00:00


In [110]:
spread_change_min=df_oil_yearly.loc[spread_change_min_year,'Spread_Change']
spread_change_max=df_oil_yearly.loc[spread_change_max_year,'Spread_Change']

print(
    f"Largest annual decrease: \n" \
    f"Year: {spread_change_min_year.year} \n" \
    f"Change: {spread_change_min:.2f}  \n\n"
    f"Largest annual increase: \n" \
    f"Year: {spread_change_max_year.year} \n" \
    f"Change: {spread_change_max:.2f}  \n\n"
)

Largest annual decrease: 
Year: 2011 
Change: -16.39  

Largest annual increase: 
Year: 2013 
Change: 7.03  




### Outlier period identification and duration analysis.


In [111]:
df_oil_yearly_summary = (
    df_oil_yearly[['Spread', 'Spread_Change']]
    .loc['2007':'2015']
    .round(2)
    .sort_values('Spread_Change')
)

display(df_oil_yearly_summary)

,Spread,Spread_Change
Date,,
2011-01-01,-16.52,-16.39
2009-01-01,0.21,-2.52
2012-01-01,-17.61,-1.09
2007-01-01,-0.15,-1.07
2010-01-01,-0.13,-0.35
2015-01-01,-3.66,2.06
2008-01-01,2.73,2.88
2014-01-01,-5.72,4.85
2013-01-01,-10.57,7.03


In [112]:
display(
    df_oil_yearly[['WTI', 'Brent', 'Spread']]
    .loc['2010':'2015']
    .round(2)
)

,WTI,Brent,Spread
Date,,,
2010-01-01,79.48,79.61,-0.13
2011-01-01,94.74,111.26,-16.52
2012-01-01,93.96,111.57,-17.61
2013-01-01,97.98,108.56,-10.57
2014-01-01,93.17,98.89,-5.72
2015-01-01,48.65,52.31,-3.66


In [113]:
df_oil_yearly[['WTI', 'Brent']].diff().loc['2010':'2015'].round(2)

,WTI,Brent
Date,,
2010-01-01,17.53,17.87
2011-01-01,15.27,31.65
2012-01-01,-0.78,0.31
2013-01-01,4.02,-3.02
2014-01-01,-4.81,-9.67
2015-01-01,-44.52,-46.58


In [114]:
df_oil_yearly['WTI_Change'] = (
    df_oil_yearly['WTI'].diff()
)

df_oil_yearly['Brent_Change'] = (
    df_oil_yearly['Brent'].diff()
)

Key Findings: WTI–Brent Spread Analysis (2007–2015)
1. During 2007–2010, the annual average WTI–Brent spread remained relatively close to zero.
2. In 2011–2012, the spread widened significantly, reaching approximately -17.61 USD/barrel in 2012. This was primarily driven by Brent prices increasing more rapidly than WTI prices in 2011.
3. In 2013, the spread narrowed considerably as WTI strengthened relative to Brent.
4. During 2014–2015, the spread continued to recover but remained negative, indicating that WTI had not fully returned to its pre-2011 relative pricing level.

In [115]:
# 1. 全部交易日的平均价差
overall_avg_spread = df_oil['Spread'].mean()

# 2. 年度价差最大下降幅度（保留正负号）
largest_annual_decrease = df_oil_yearly['Spread_Change'].min()

# 3. 年度价差最大上升幅度
largest_annual_increase = df_oil_yearly['Spread_Change'].max()

# 4. 低于 IQR 下界的记录数量
lower_outlier_count = (df_oil['Spread'] < lower_bound).sum()

print(overall_avg_spread, largest_annual_decrease, largest_annual_increase, lower_outlier_count)

-1.7080704800817161 -16.387197260624678 7.033925224708357 559


### Interactive dashboard development using Python and Plotly.

In [116]:
fig = make_subplots(
    rows=3,
    cols=4,
    shared_xaxes=True,

    specs=[
        [{'type': 'indicator'}, {'type': 'indicator'},
         {'type': 'indicator'}, {'type': 'indicator'}],

        [{'type': 'xy', 'colspan': 4}, None, None, None],

        [{'type': 'xy', 'colspan': 4}, None, None, None]
    ],

    row_heights=[0.20, 0.40, 0.40],
    horizontal_spacing=0.08,
    vertical_spacing=0.12
)

# KPI 1: Average Spread
fig.add_trace(
    go.Indicator(
        mode='number',
        value=overall_avg_spread,
        title={
            'text': 'Average Spread',
            'font': {'size': 18}
        },
        number={
            'suffix': ' USD/bbl',
            'valueformat': '.2f',
            'font': {'size': 28}
        }
    ),
    row=1, col=1
)

# KPI 2: Largest Annual Decrease
fig.add_trace(
    go.Indicator(
        mode='number',
        value=largest_annual_decrease,
        title={
            'text': 'Largest Decrease',
            'font': {'size': 18}
        },
        number={
            'suffix': ' USD/bbl',
            'valueformat': '.2f',
            'font': {'size': 28}
        }
    ),
    row=1, col=2
)

# KPI 3: Largest Annual Increase
fig.add_trace(
    go.Indicator(
        mode='number',
        value=largest_annual_increase,
        title={
            'text': 'Largest Increase',
            'font': {'size': 18}
        },
        number={
            'suffix': ' USD/bbl',
            'valueformat': '.2f',
            'font': {'size': 28}
        }
    ),
    row=1, col=3
)

# KPI 4: Lower Outliers
fig.add_trace(
    go.Indicator(
        mode='number',
        value=lower_outlier_count,
        title={
            'text': 'Lower Outliers',
            'font': {'size': 18}
        },
        number={
            'valueformat': ',.0f',
            'font': {'size': 28}
        }
    ),
    row=1, col=4
)

fig.update_layout(
    width=1400,
    height=1000,
    margin=dict(l=50, r=50, t=100, b=50)
)

# Chart 1: Annual Average WTI-Brent Spread
fig.add_trace(
    go.Scatter(
        x=df_oil_yearly.index,
        y=df_oil_yearly['Spread'],
        mode='lines+markers',
        name='Spread',
        line=dict(color='green', width=2),
        hovertemplate=(
            'Year: %{x|%Y}<br>'
            'Spread: %{y:.2f} USD/barrel'
            '<extra></extra>'
        )
    ),
    row=2, col=1
)

# Zero reference line for annual average spread
fig.add_trace(
    go.Scatter(
        x=df_oil_yearly.index,
        y=[0] * len(df_oil_yearly),
        mode='lines',
        name='Zero Line',
        line=dict(color='gray', dash='dash', width=1),
        showlegend=False,
        hoverinfo='skip'
    ),
    row=2, col=1
)

# Add vertical highlight region
fig.add_vrect(
    x0='2011-01-01',
    x1='2013-12-31',
    fillcolor='red',
    opacity=0.15,
    line_width=0,
    row=2,
    col=1,
    exclude_empty_subplots=False,             ## 对于包含 go.Indicator 和 XY 子图的混合 Dashboard时使用
    annotation_text='Abnormal Spread Period',
    annotation_position='top left'  
)

min_spread_yearly=df_oil_yearly.loc[df_oil_yearly['Spread'].idxmin()]
#print(min_spread_yearly)
#print(min_spread_yearly.name)
#print(min_spread_yearly['Spread'])


fig.add_annotation(
    x=min_spread_yearly.name,
    y=min_spread_yearly['Spread'],
    text=f"Min Spread: {min_spread_yearly['Spread']:.2f}",
    showarrow=True,
    arrowhead=5,
    ax=80,
    ay=-40,
    row=2,
    col=1
)

# Chart 2: Annual Change in WTI
fig.add_trace(
    go.Scatter(
        x=df_oil_yearly.index,
        y=df_oil_yearly['WTI_Change'],
        mode='lines+markers',
        name='WTI Change',
        line=dict(color='royalblue', width=2),
        hovertemplate=(
            'Year: %{x|%Y}<br>'
            'WTI Change: %{y:.2f} USD/barrel'
            '<extra></extra>'
        )
    ),
    row=3, col=1
)

# Chart 3: Annual Change in Brent
fig.add_trace(
    go.Scatter(
        x=df_oil_yearly.index,
        y=df_oil_yearly['Brent_Change'],
        mode='lines+markers',
        name='Brent Change',
        line=dict(color='darkorange', width=2),
        hovertemplate=(
            'Year: %{x|%Y}<br>'
            'Brent Change: %{y:.2f} USD/barrel'
            '<extra></extra>'
        )
    ),
    row=3, col=1
)

# Zero reference line for annual change
fig.add_trace(
    go.Scatter(
        x=df_oil_yearly.index,
        y=[0] * len(df_oil_yearly),
        mode='lines',
        name='Zero Line',
        line=dict(color='gray', dash='dash', width=1),
        showlegend=False,
        hoverinfo='skip'
    ),
    row=3, col=1
)

fig.update_layout(
    title={
        'text': 'WTI-Brent Analysis Dashboard',
        'font': {'size': 24},
        'x': 0.5,
        'xanchor': 'center'
    },
    width=1400,
    height=1000,
    template='plotly_white',
    showlegend=True,
    hovermode='x unified',
    margin=dict(l=50, r=50, t=100, b=50),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=0.68,
        xanchor='right',
        x=1
    )
)

fig.update_xaxes(title_text='Year', row=3, col=1)

fig.update_yaxes(
    title_text='Spread (USD/barrel)',
    row=2, col=1
)

fig.update_yaxes(
    title_text='Price Change (USD/barrel)',
    row=3, col=1
)

fig.show()

*Case Study: The Negative WTI Oil Price Event — April 20, 2020*


In [117]:
df_oil.loc[
    (df_oil['Date'] >= '2020-04-15') &
    (df_oil['Date'] <= '2020-04-24')
].sort_values('Date')

,Date,WTI,Brent,Spread
8223,2020-04-15,19.96,19.80,0.16
8224,2020-04-16,19.82,18.69,1.13
8225,2020-04-17,18.31,19.75,-1.44
8226,2020-04-20,-36.98,17.36,-54.34
8227,2020-04-21,8.91,9.12,-0.21
8228,2020-04-22,13.64,13.77,-0.13
8229,2020-04-23,15.06,15.06,0.00
8230,2020-04-24,15.99,15.87,0.12


In [118]:
df_oil = df_oil.sort_values('Date').reset_index(drop=True)
df_oil['WTI_Change']=df_oil['WTI'].diff()
df_oil['Brent_Change']=df_oil['Brent'].diff()
df_oil['Change_Difference']=df_oil['WTI_Change']-df_oil['Brent_Change']
df_oil['Spread_Change']=df_oil['Spread'].diff()

In [119]:
(df_oil['Spread_Change'] - df_oil['Change_Difference']).abs().max()

np.float64(7.105427357601002e-15)

### Case Study: The April 2020 Negative WTI Price Event

On April 20, 2020, the WTI price in the dataset dropped sharply to -$36.98 per barrel, while Brent remained positive at $17.36 per barrel. Consequently, the WTI-Brent spread reached -$54.34 per barrel, representing a one-day deterioration of $52.90 per barrel compared with the previous observation.

The event was short-lived. On April 21, the WTI price recovered to $8.91 per barrel, and the spread narrowed to -$0.21 per barrel.

Several market factors contributed to this exceptional event. The COVID-19 pandemic caused an unprecedented decline in global oil demand, while crude oil inventories continued to increase. Storage availability at Cushing, Oklahoma, the physical delivery location for WTI futures, became increasingly constrained. At the same time, the May 2020 WTI futures contract was approaching expiration on April 21. Traders without the ability or arrangements to take physical delivery faced strong pressure to close their positions. Limited liquidity and insufficient buying interest contributed to the extreme negative price movement.

This event demonstrates that the WTI-Brent spread can be affected not only by differences in regional crude oil supply and demand, but also by futures contract structures, delivery requirements, storage constraints, and market liquidity.

**Key takeaway:** The extreme negative spread observed on April 20, 2020, was an exceptional market event associated with the expiration of the WTI May futures contract, rather than evidence that crude oil in general had become worthless.

**Data limitation:** The prices shown above are taken from the dataset used in this project. They should not automatically be interpreted as the official settlement prices of the May 2020 WTI futures contract. The exact price-series definitions and contract specifications should be verified before making a direct comparison with official futures settlement data.

## 4. Key Findings

### Finding 1: Long-term Spread Trend

Before 2011, the annual average WTI-Brent spread was relatively close to zero. The spread moved sharply lower in 2011 and reached its most negative annual average in 2012 (-17.61 USD/bbl). It then recovered in 2013, but remained negative through 2015.

### Finding 2: Concentration of Lower Outliers

The IQR method identified 559 lower outliers and 4 upper outliers in the merged daily dataset. Of the 559 lower outliers, 496 (88.7%) occurred during 2011-2013. These 496 observations represented 66.2% of all 749 merged observations in that three-year period.

### Finding 3: Persistent Outlier Periods

The 496 lower-outlier observations in 2011-2013 were concentrated in three long episodes. The three episodes contained 431 observations (86.9% of the period's lower outliers), indicating persistence rather than a uniform distribution of isolated observations.

### Finding 4: Relative Price Movements

The change in the spread is the difference between the changes in WTI and Brent: ΔSpread = ΔWTI - ΔBrent. In 2011, Brent increased more than WTI, pushing the spread lower. In 2012, WTI decreased while Brent increased slightly, widening the discount further. In 2013, WTI increased while Brent decreased, contributing to a substantial recovery in the spread.

### Finding 5: April 2020 Negative WTI Spot Price Observation

In the DataHub/EIA-derived spot-price series used here, WTI was -36.98 USD/bbl on April 20, 2020, while Brent was 17.36 USD/bbl. The spread reached -54.34 USD/bbl and improved to -0.21 USD/bbl on April 21. This was an exceptional observation. The market context included the pandemic-driven demand shock, constrained storage availability at Cushing, and stress surrounding the expiry of the May 2020 WTI futures contract. The dataset values are spot-price observations and should not be described as the official futures settlement price.

## 5. Limitations

- The analysis uses a historical DataHub/DataPressr snapshot of EIA daily spot-price series. Later revisions or differences in data snapshots may affect historical coverage.
- The inner join retains only dates with both WTI and Brent observations; dates missing from either series are excluded.
- IQR identifies statistical outliers, not necessarily erroneous observations. Market context is required before assigning an economic explanation.
- The 2026 data end on September 1 in the notebook snapshot and represent a partial year. They should not be compared directly with complete calendar years.
- The analysis is descriptive. It identifies historical patterns and co-movements but does not establish causality.

## 6. Conclusion

The WTI-Brent spread experienced a pronounced structural shift during 2011-2013, with lower outliers concentrated in several persistent episodes. Annual price movements show how the relative performance of the two benchmarks translated into changes in the spread. The April 2020 observation provides a separate example of an extreme market dislocation. Together, the analysis demonstrates an end-to-end workflow for energy-market data preparation, quality control, statistical analysis, time-series aggregation, outlier episode detection, and interactive visualization.

This notebook is an exploratory historical analysis, not an investment recommendation or a causal model.